## Imports

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean
import cartopy.crs as ccrs
import tqdm
import copy
from sklearn.preprocessing import PowerTransformer
import sklearn.linear_model

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_windowed(data, window_size=480, stride=60):
    """Get windowed version of data (used for computing parameter values over time)"""

    ## get windowed data
    data_rolling = data.rolling({"time": window_size}, center=True)
    data_windowed = data_rolling.construct(window_dim="sample", stride=stride)

    ## Remove NaN values
    if type(data) is xr.Dataset:
        da = data_windowed.to_dataarray().isel(variable=0)
    else:
        da = data_windowed
    valid_idx = ~np.isnan(da).any(["sample", "member"])
    data_windowed = data_windowed.isel(time=valid_idx)

    ## drop variable name
    if type(data) is xr.Dataset:
        data_windowed = data_windowed.drop_vars("variable")

    ## rename window coord from time to year
    year_coord = dict(year=data_windowed.time.dt.year.values)
    data_windowed = data_windowed.rename({"time": "year"}).assign_coords(year_coord)

    ## rename sample coord to time (use arbitrary time, used for seasonality)
    time_coord = dict(
        time=xr.date_range(start="1850-01-01", freq="MS", periods=window_size)
    )
    data_windowed = data_windowed.rename({"sample": "time"}).assign_coords(time_coord)

    return data_windowed


def fit_pt(z, get_lambda=False):
    """fit power transform to data"""
    if np.isnan(z).all():
        return np.nan

    else:
        ## fit transform
        pt = PowerTransformer(standardize=True).fit(z)

        if get_lambda:
            return pt, pt.lambdas_.item()

        else:
            return pt


def pt_mean(X):
    """find power-transformed mean"""

    ## transpose X
    X_ = copy.deepcopy(X.transpose("time", "member"))

    ## do transform
    pt = fit_pt(X_.values)

    ## empty array to hold inverse results
    Y = copy.deepcopy(X_).isel(time=0).drop_vars("time")

    ## invert mean
    Y.values = pt.inverse_transform(np.zeros([1, len(X_.member)])).squeeze()
    return Y


def plot_quantiles(ax, data, varname, label=None, **plot_kwargs):
    """plot ensemble quantiles for data on given ax object"""

    ## loop thru quantiles
    for q, lw, label in zip([0.5, 0.1, 0.9], [2, 0.5, 0.5], [label, None, None]):

        ## plot data
        ax.plot(
            data.year,
            norm(data[varname]).quantile(q=q, dim="member"),
            lw=lw,
            label=label,
            **plot_kwargs,
        )

    return

## Load data

In [ ]:
USE_WIDE = True

## load spatial data
forced, anom_ = src.utils.load_consolidated()

if USE_WIDE:

    ## load wide data
    forced_05, anom_05 = src.utils.load_consolidated_05()

    for v in list(forced_05):
        forced[v] = forced_05[v]
        anom_[v] = anom_05[v]

## get subset of data for total
VARNAMES = ["T", "sst", "taux", "tauy", "w"]
total = anom_[VARNAMES] + forced[VARNAMES]

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th = Th_total - Th_total.mean("member")

## compute dTdx

In [ ]:
## compute diff. def of Tw
sel_lat = lambda x: x.sel(latitude=slice(-5, 5)).mean(["latitude", "longitude"])
get_Tw = lambda x: sel_lat(x.sel(longitude=slice(140, 190)))
get_Te = lambda x: sel_lat(x.sel(longitude=slice(240, 280)))
get_dTdx = lambda x: get_Tw(x) - get_Te(x)

## compute dTdx (annual mean)
dTdx = src.utils.reconstruct_wrapper(
    xr.merge([total["sst"], forced["sst_comp"]]), get_dTdx
)
dTdx = dTdx.rename({"sst": "dTdx"})
data = xr.merge([Th["T_34"], dTdx])

## window
data = get_windowed(data, window_size=480, stride=12)

In [ ]:
# norm = lambda

## specify variable
v = "dTdx"

n_avg = 20
dTdx_mean0, _ = src.utils.separate_forced(dTdx[v], use_mean=True, n=0)
dTdx_mean, _ = src.utils.separate_forced(dTdx[v], use_mean=True, n=n_avg)
dTdx_median, _ = src.utils.separate_forced(dTdx[v], use_mean=False, n=n_avg)

sel = lambda x: x.groupby("time.year").mean()

fig, ax = plt.subplots(figsize=(3, 2.5))
# ax.plot(sel(dTdx_mean0).year, sel(dTdx_mean0))
ax.plot(sel(dTdx_mean).year, sel(dTdx_mean))
# ax.plot(sel(dTdx_median).year, sel(dTdx_median)-.05)
ax.plot(data.year, data[v].mean("time").median("member"))
ax.plot(data.year, data[v].median("time").median("member") - 0.15)
# ax.plot(T34_mean, zorder=10)
# ax.plot(T34_mean0[n_avg:-n_avg])
# plt.show()
ax.axvline(2010)
plt.show()

## Compare methods

### Mean/median

In [ ]:
data["dTdx_mean"] = data["dTdx"].mean("time")
data["dTdx_median"] = data["dTdx"].median("time")
data["sigma_T"] = data["T_34"].std("time")

### Power transform

update this: find mean in transformed coordinates, then invert to get correction

In [ ]:
dTdx_pt = []
for year in tqdm.tqdm(data.year):
    dTdx_pt.append(pt_mean(data["dTdx"].sel(year=year)))
data["dTdx_pt"] = xr.concat(dTdx_pt, dim=data.year)

## transform data for all years at the same time
# data["dTdx_pt_all"] = fit_transform_pt(data["dTdx"]).mean("time")

### ENSO regression

In [ ]:
## shared args
kwargs = dict(data=data, xvar="sigma_T", yvars=["dTdx_mean"])

## remove effect
data["dTdx_norect"] = src.utils.remove_rect(**kwargs)["dTdx_mean"]
data["dTdx_norect0"] = src.utils.remove_rect(constant=True, **kwargs)["dTdx_mean"]

## compare to other method
coefs0 = src.utils.estimate_rect(data, xvar="sigma_T", yvars=["dTdx_mean"]).sel(
    degree=0
)["dTdx_mean"]

## check equivalence
print(np.allclose(data["dTdx_norect"].mean("member"), coefs0))

### Plot

In [ ]:
## func to normalize data
norm = lambda x: x - x.isel(year=0).median("member")


## colors
colors = sns.color_palette("colorblind")

fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.5), layout="constrained")

## plot mean and median
for i, (x, label, color) in enumerate(
    zip(
        ["dTdx_mean", "dTdx_median"],
        ["Mean", "Median"],
        ["k", colors[0]],
    )
):
    plot_quantiles(axs[0], data, x, c=color, label=label)

## plot power transform
for i, (x, label, color) in enumerate(
    zip(
        ["dTdx_mean", "dTdx_pt"],  # , "dTdx_pt_all"],
        [None, "PT", "PT (const.)"],
        ["k", *colors[1:3]],
    )
):
    plot_quantiles(axs[1], data, x, c=color, label=label)

## plot nonrect
for i, (x, label, color) in enumerate(
    zip(
        ["dTdx_mean", "dTdx_norect", "dTdx_norect0"],
        [None, "No-rect", "No-rect (const.)"],
        ["k", colors[3], colors[5]],
    )
):
    plot_quantiles(axs[2], data, x, c=color, label=label)


## formatting
src.utils.set_lims(axs)
for ax in axs:
    ax.axvline(2010, c="k", ls="--", lw=0.8)
    ax.axhline(0, c="k", ls="--", lw=0.8)
    ax.set_xticks([1870, 2010, 2080])
    ax.legend(loc="lower left")
    # ax.axhline(0.4, ls="--")

plt.show()